In [1]:
import os
from pathlib import Path
import math
import random
import joblib
from collections import Counter, defaultdict

import numpy as np
from PIL import Image, ImageOps

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support


In [2]:
# Folder that contains class subfolders
DATA_DIR = Path("Face_Data_cropped")     

# images will become 64x64 (grayscale)
IMG_SIZE = 64                 
GRAYSCALE = True              

# Candidate k values for manual tuning
K_CANDIDATES = [1, 3, 5, 7, 9]

In [3]:
def load_image(path: Path, grayscale=True, img_size=64) -> np.ndarray:
    # open image -> converst to rgb -> converts to grayscale -> resize -> convert to numpy array -> scaling 
    img = Image.open(path).convert("RGB")
    if grayscale:
        img = img.convert("L")  # single channel
    img = img.resize((img_size, img_size), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32)
    
    arr = arr / 255.0
    return arr

def to_feature_vector(img_array: np.ndarray) -> np.ndarray:
    # converts 2D to 1D
    return img_array.flatten()

def list_image_files(root: Path):
    # reurtn all the images from a folder
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for p in root.rglob("*"):
        if p.suffix.lower() in exts:
            yield p


In [4]:
# Build dataset matrices from DATA_DIR (expects DATA_DIR/<person_id>/*.jpg ...)
def build_dataset(root: Path) -> tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    for cls_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        label = cls_dir.name
        for img_path in list_image_files(cls_dir):
            img_arr = load_image(img_path, grayscale=GRAYSCALE, img_size=IMG_SIZE)  # [H,W] in 0..1
            X.append(to_feature_vector(img_arr))  # flatten -> (D,)
            y.append(label)
    return np.vstack(X).astype(np.float64), np.array(y)

X, y = build_dataset(DATA_DIR)
print("X :", X.shape, " | classes:", len(np.unique(y)))


X : (178, 4096)  | classes: 10


In [5]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y, random_state = 42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size = 0.125, stratify = y_temp, random_state = 42
)

print("Train: ",X_train.shape, "Val: ", X_val.shape, "Test: ", X_test.shape)

Train:  (124, 4096) Val:  (18, 4096) Test:  (36, 4096)


In [13]:
import numpy as np

class PCAFromScratch:
    def __init__(self, n_components=None, keep_variance=None, whiten=False):
        
        assert (n_components is None) ^ (keep_variance is None), "Set exactly one of n_components or keep_variance."
        self.n_components = n_components
        self.keep_variance = keep_variance
        self.whiten = whiten
        # learned params
        self.mean_ = None
        self.components_ = None      # shape (n_comp, D)
        self.eigvals_ = None
        self.explained_variance_ratio_ = None

    def fit(self, X):
      
        # center
        self.mean_ = X.mean(axis=0, keepdims=True)   # (1, D)
        Xc = X - self.mean_

        # use SVD (stable): Xc = U S Vt, PCs = rows of Vt
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
        eigvals = (S**2) / (X.shape[0] - 1)         # eigenvalues of covariance
        total_var = eigvals.sum()
        ratio = eigvals / total_var

        if self.keep_variance is not None:
            cumsum = np.cumsum(ratio)
            k = np.searchsorted(cumsum, self.keep_variance) + 1
        else:
            k = int(self.n_components)

        self.components_ = Vt[:k, :]                # (k, D)
        self.eigvals_ = eigvals[:k]
        self.explained_variance_ratio_ = ratio[:k]
        return self

    def transform(self, X):
        Xc = X - self.mean_
        Z = Xc @ self.components_.T                 # (N, k)
        if self.whiten:
            Z = Z / np.sqrt(self.eigvals_ + 1e-12)
        return Z

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    @property
    def n_comp_(self):
        return 0 if self.components_ is None else self.components_.shape[0]


In [14]:
# keep 95% variance
pca = PCAFromScratch(keep_variance=0.95, whiten=False)

X_train_pca = pca.fit_transform(X_train)   # fit on train only
X_val_pca   = pca.transform(X_val)         # transform val with the same PCA

# DO NOT TOUCH TEST YET — we’ll only transform test in the final step.
print("Previos components kept:", X_train.shape[1])
print("PCA components kept:", pca.n_comp_)
print("Explained variance used: ~", round(pca.explained_variance_ratio_.sum()*100, 2), "%")


Previos components kept: 4096
PCA components kept: 62
Explained variance used: ~ 95.02 %


In [15]:
from collections import Counter
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def knn_predict_one(xq, X_ref, y_ref, k=3):
    # Euclidean distance (fast with vectorization)
    d = np.linalg.norm(X_ref - xq, axis=1)
    idx = np.argpartition(d, k-1)[:k]
    labels = y_ref[idx]
    # majority vote; break ties by smallest average distance
    counts = Counter(labels)
    maxc = max(counts.values())
    cands = [c for c, v in counts.items() if v == maxc]
    if len(cands) == 1:
        return cands[0]
    # tie-breaker
    best_lbl, best_d = None, 1e18
    for lbl in cands:
        m = (labels == lbl)
        avg = d[idx][m].mean()
        if avg < best_d:
            best_lbl, best_d = lbl, avg
    return best_lbl

def knn_predict_batch(Xq, X_ref, y_ref, k=3):
    return np.array([knn_predict_one(x, X_ref, y_ref, k=k) for x in Xq])

def report(y_true, y_pred, title=""):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    print(f"{title} — acc={acc:.4f} | macro P={prec:.4f} R={rec:.4f} F1={f1:.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

# quick baseline with k=3
y_val_pred_knn = knn_predict_batch(X_val_pca, X_train_pca, y_train, k=3)
report(y_val, y_val_pred_knn, title="KNN(k=3) on VAL")


KNN(k=3) on VAL — acc=0.7222 | macro P=0.6667 R=0.7500 F1=0.6800
Confusion matrix:
 [[2 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 0]
 [0 0 1 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 1 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 2 0]
 [0 0 1 0 0 0 0 0 0 1]]


In [20]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', C=1.0, gamma='scale')  # sensible defaults
svm.fit(X_train_pca, y_train)
y_val_pred_svm = svm.predict(X_val_pca)
report(y_val, y_val_pred_svm, title="SVM(RBF, C=1, gamma='scale') on VAL")


SVM(RBF, C=1, gamma='scale') on VAL — acc=0.7222 | macro P=0.6167 R=0.7500 F1=0.6433
Confusion matrix:
 [[2 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 0]
 [0 0 1 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 0 1 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 2 0]
 [0 0 1 0 0 0 0 0 0 1]]


In [21]:
from sklearn.metrics import f1_score, accuracy_score

# ---- KNN tuning (by macro F1) ----
best_knn = {"k": None, "f1": -1.0, "acc": -1.0, "pred": None}
for k in K_CANDIDATES:
    pred = knn_predict_batch(X_val_pca, X_train_pca, y_train, k=k)
    f1  = f1_score(y_val, pred, average="macro")
    acc = accuracy_score(y_val, pred)
    # pick by F1 first; if tie, prefer higher accuracy
    if (f1 > best_knn["f1"]) or (f1 == best_knn["f1"] and acc > best_knn["acc"]):
        best_knn = {"k": k, "f1": f1, "acc": acc, "pred": pred}

print(f"[TUNE] Best KNN -> k={best_knn['k']} | macro-F1={best_knn['f1']:.4f} | acc={best_knn['acc']:.4f}")
report(y_val, best_knn["pred"], title=f"KNN(k={best_knn['k']}) on VAL")

[TUNE] Best KNN -> k=1 | macro-F1=0.7038 | acc=0.7222
KNN(k=1) on VAL — acc=0.7222 | macro P=0.7067 R=0.7500 F1=0.7038
Confusion matrix:
 [[2 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 0]
 [0 0 1 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 1 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 2 0]
 [0 0 0 0 0 0 0 0 1 1]]


In [23]:
# ---- SVM tuning (by macro F1) ----
C_GRID     = [0.1, 1, 3, 10, 30, 100]
GAMMA_GRID = ["scale", 0.001, 0.003, 0.01, 0.03, 0.1]

best_svm = {"C": None, "gamma": None, "f1": -1.0, "acc": -1.0, "model": None, "pred": None}
for C in C_GRID:
    for gamma in GAMMA_GRID:
        clf = SVC(kernel='rbf', C=C, gamma=gamma)
        clf.fit(X_train_pca, y_train)
        pred = clf.predict(X_val_pca)
        f1  = f1_score(y_val, pred, average="macro")
        acc = accuracy_score(y_val, pred)
        if (f1 > best_svm["f1"]) or (f1 == best_svm["f1"] and acc > best_svm["acc"]):
            best_svm = {"C": C, "gamma": gamma, "f1": f1, "acc": acc, "model": clf, "pred": pred}

print(f"[TUNE] Best SVM -> C={best_svm['C']}, gamma={best_svm['gamma']} | macro-F1={best_svm['f1']:.4f} | acc={best_svm['acc']:.4f}")
report(y_val, best_svm["pred"], title=f"SVM(C={best_svm['C']}, gamma={best_svm['gamma']}) on VAL")

[TUNE] Best SVM -> C=30, gamma=0.001 | macro-F1=0.6767 | acc=0.7222
SVM(C=30, gamma=0.001) on VAL — acc=0.7222 | macro P=0.6667 R=0.7500 F1=0.6767
Confusion matrix:
 [[2 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 0]
 [0 0 1 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 1 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 2 0]
 [0 0 1 0 0 0 0 0 0 1]]


In [24]:
# 1) Merge train+val
X_trval = np.vstack([X_train, X_val])
y_trval = np.hstack([y_train, y_val])

# 2) Refit PCA on train+val using the SAME dimensionality chosen earlier
fixed_n = pca.n_comp_                       # use #components to avoid "variance peeking"
pca_final = PCAFromScratch(n_components=fixed_n, whiten=False)
X_trval_pca = pca_final.fit_transform(X_trval)
X_test_pca  = pca_final.transform(X_test)   # transform test once, at the very end

# 3) Train final KNN
knn_final_k = best_knn["k"]
def knn_predict_batch_ref(Xq):
    return np.array([knn_predict_one(x, X_trval_pca, y_trval, k=knn_final_k) for x in Xq])
y_test_pred_knn = knn_predict_batch_ref(X_test_pca)
print("\n=== FINAL KNN on TEST ===")
report(y_test, y_test_pred_knn, title=f"KNN(k={knn_final_k}) on TEST")

# 4) Train final SVM
svm_final = SVC(kernel='rbf', C=best_svm["C"], gamma=best_svm["gamma"])
svm_final.fit(X_trval_pca, y_trval)
y_test_pred_svm = svm_final.predict(X_test_pca)
print("\n=== FINAL SVM on TEST ===")
report(y_test, y_test_pred_svm, title=f"SVM(C={best_svm['C']}, gamma={best_svm['gamma']}) on TEST")



=== FINAL KNN on TEST ===
KNN(k=1) on TEST — acc=0.9444 | macro P=0.9550 R=0.9417 F1=0.9403
Confusion matrix:
 [[4 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 1 0]
 [0 0 4 0 0 0 0 0 0 0]
 [0 0 0 4 0 0 0 0 0 0]
 [0 0 0 0 3 0 0 0 0 0]
 [0 0 0 0 0 4 0 0 0 0]
 [0 0 0 0 0 0 3 0 0 0]
 [0 0 0 0 1 0 0 2 0 0]
 [0 0 0 0 0 0 0 0 4 0]
 [0 0 0 0 0 0 0 0 0 3]]

=== FINAL SVM on TEST ===
SVM(C=30, gamma=0.001) on TEST — acc=0.9444 | macro P=0.9550 R=0.9500 F1=0.9460
Confusion matrix:
 [[4 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 1 0 0 0 0 0]
 [0 0 4 0 0 0 0 0 0 0]
 [0 0 0 4 0 0 0 0 0 0]
 [0 0 0 0 3 0 0 0 0 0]
 [0 0 0 0 0 4 0 0 0 0]
 [0 0 0 0 0 0 3 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [1 0 0 0 0 0 0 0 3 0]
 [0 0 0 0 0 0 0 0 0 3]]


In [15]:
import joblib

joblib.dump(pca_final, "pca_final.pkl")
joblib.dump({"k": knn_final_k, "X_ref": X_trval_pca, "y_ref": y_trval}, "knn_final_refs.pkl")
joblib.dump(svm_final, "svm_final.pkl")
print("Saved pca_final.pkl, knn_final_refs.pkl, svm_final.pkl")


Saved pca_final.pkl, knn_final_refs.pkl, svm_final.pkl


In [25]:
# --- EXACT same preprocessing as training: MTCNN crop -> grayscale 64x64 -> [0,1] -> flatten ---

from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
import cv2, joblib
from mtcnn import MTCNN

# 1) Face detector (same as training)
detector = MTCNN()

# 2) Load RGB with orientation fix
def load_rgb_with_exif(path: Path) -> np.ndarray:
    img = Image.open(path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    return np.array(img)  # HxWx3, uint8

# 3) Crop largest face with padding (must match training)
def crop_face(image_rgb, min_side=40, out_size=160):
    faces = detector.detect_faces(image_rgb)
    if not faces:
        return None
    # largest face
    faces.sort(key=lambda f: f['box'][2] * f['box'][3], reverse=True)
    x, y, w, h = faces[0]['box']
    if w < min_side or h < min_side:
        return None
    pad = int(0.25 * max(w, h))  # <- keep this the SAME as training
    H, W = image_rgb.shape[:2]
    x1 = max(0, x - pad); y1 = max(0, y - pad)
    x2 = min(W, x + w + pad); y2 = min(H, y + h + pad)
    face = image_rgb[y1:y2, x1:x2]
    if face.size == 0:
        return None
    return cv2.resize(face, (out_size, out_size), interpolation=cv2.INTER_AREA)

# 4) Convert cropped RGB face -> grayscale(64x64) -> float32 [0,1]
def to_gray64_float01(face_rgb: np.ndarray, out_size=64) -> np.ndarray:
    pil = Image.fromarray(face_rgb)     # RGB
    pil = pil.convert("L")              # grayscale
    pil = pil.resize((out_size, out_size), Image.BILINEAR)
    arr = np.array(pil, dtype=np.float32) / 255.0    # (64,64) in [0,1]
    return arr

# 5) Flatten to feature vector (4096,)
def to_feature_vector(img_array: np.ndarray) -> np.ndarray:
    return img_array.flatten()

# === FINAL: preprocess_one(path) that replicates training exactly ===
def preprocess_one(path: str | Path) -> np.ndarray:
    """
    Path -> load RGB (with EXIF fix) -> MTCNN crop (pad=0.25, out=160) ->
    grayscale 64x64 -> [0,1] -> flatten -> float64
    Returns shape: (4096,)
    """
    rgb = load_rgb_with_exif(Path(path))
    face = crop_face(rgb, min_side=40, out_size=160)
    if face is None:
        raise ValueError("No suitable face detected for this image.")
    arr = to_gray64_float01(face, out_size=64)       # (64,64)
    x = to_feature_vector(arr).astype(np.float64)    # (4096,)
    return x


In [26]:
def knn_predict_one_with_score(xq, X_ref, y_ref, k=3, use_min=False):
    """
    Returns:
      pred_label: majority-vote label (tie broken by smallest avg distance)
      score: distance-based confidence (lower = more confident)
             mean or min distance among neighbors of the predicted label
    """
    d = np.linalg.norm(X_ref - xq, axis=1)
    idx = np.argpartition(d, k-1)[:k]
    labels = y_ref[idx]

    # majority vote
    counts = Counter(labels)
    maxc = max(counts.values())
    cands = [c for c, v in counts.items() if v == maxc]

    if len(cands) == 1:
        pred = cands[0]
    else:
        # tie-breaker: smallest average distance
        best_lbl, best_d = None, 1e18
        for lbl in cands:
            m = (labels == lbl)
            avg = d[idx][m].mean()
            if avg < best_d:
                best_lbl, best_d = lbl, avg
        pred = best_lbl

    # score for thresholding
    m = (labels == pred)
    score = float(d[idx][m].min() if use_min else d[idx][m].mean())
    return pred, score


In [27]:
import joblib
from sklearn.metrics import accuracy_score, f1_score

PERCENTILE = 95  # tweak to taste; try 90, 95, 98

scores_correct = []
preds = []

for i in range(X_val_pca.shape[0]):
    pred, score = knn_predict_one_with_score(X_val_pca[i], X_trval_pca, y_trval, k=knn_k)
    preds.append(pred)
    if pred == y_val[i]:
        scores_correct.append(score)

val_acc = accuracy_score(y_val, preds)
val_f1  = f1_score(y_val, preds, average="macro")

tau_max = float(np.max(scores_correct)) if scores_correct else float("inf")
tau_pct = float(np.percentile(scores_correct, PERCENTILE)) if scores_correct else float("inf")

print(f"VAL (no-unknown) baseline: acc={val_acc:.4f} | macro-F1={val_f1:.4f}")
print(f"Chosen τ at {PERCENTILE}th percentile: {tau_pct:.6f}   (max-correct τ: {tau_max:.6f})")

# Save the threshold you want to use
tau = tau_pct   # or tau_max for strict mode
joblib.dump(tau, "threshold.pkl")
print("Saved threshold.pkl")


VAL (no-unknown) baseline: acc=0.0556 | macro-F1=0.0250
Chosen τ at 95th percentile: 10.270593   (max-correct τ: 10.270593)
Saved threshold.pkl


In [31]:
# Load artifacts as you already do
pca_final = joblib.load("pca_final.pkl")
svm_final = joblib.load("svm_final.pkl")
tauP = joblib.load("threshold.pkl")
tauP = tauP+.8

# If you use the reference-based KNN in PCA space:
knn_refs  = joblib.load("knn_final_refs.pkl")  # {"k":..., "X_ref":..., "y_ref":...}
knn_k, X_ref, y_ref = knn_refs["k"], knn_refs["X_ref"], knn_refs["y_ref"]

def knn_pred_and_score(xq, X_ref, y_ref, k):
    """
    Returns:
      pred_label: KNN majority label
      score: mean distance of neighbors that match pred_label  (lower is better)
    """
      # 1. compute distances to all reference samples
    d = np.linalg.norm(X_ref - xq, axis=1)
    
    # 2. get indices of the k nearest neighbors
    idx = np.argpartition(d, k)[:k]
    
    # 3. slice distances and labels for those neighbors
    d_k = d[idx]
    lbl_k = y_ref[idx]
    # majority vote
    vals, counts = np.unique(lbl_k, return_counts=True)
    pred = vals[np.argmax(counts)]
    # mean distance among those neighbors that match the predicted label
    same = (lbl_k == pred)
    score = float(d_k[same].mean())
    return pred, score

def predict_image(path: str | Path):
    x = preprocess_one(path).reshape(1, -1)    # (1,4096)
    z = pca_final.transform(x)                 # (1,k)  IMPORTANT: do NOT refit
    pred_svm = svm_final.predict(z)[0]
    pred_knn, score = knn_pred_and_score(z[0], X_ref, y_ref, k=knn_k)

    print("score: ", score)

    if score <= tauP:
        pred_knn = pred_knn
    else: 
        pred_knn = "unknown_label"
        
    return {"svm": pred_svm, "knn": pred_knn}

res = predict_image("test4.jpg")
print(res)  # {'svm': 'PersonA', 'knn': 'PersonA'}


score:  8.534946908492405
{'svm': 'Zuhayr_6', 'knn': 'Elon_musk_5'}
